In [1]:
import duckdb

conn = duckdb.connect('base_clinica.duckdb')

In [2]:
conn.execute("""
    SELECT Id AS patient_id, COUNT(*) AS repeticiones
    FROM patients
    GROUP BY Id
    HAVING COUNT(*) > 1;
""").df()

,patient_id,repeticiones


In [3]:
conn.execute("""
    SELECT e.Id AS encounter_id, e.PATIENT, p.BIRTHDATE, e.START AS fecha_encuentro
    FROM encounters e
    JOIN patients p ON e.PATIENT = p.Id
    WHERE e.START < p.BIRTHDATE;
""").df()

,encounter_id,PATIENT,BIRTHDATE,fecha_encuentro


In [4]:
conn.execute("""
    SELECT 
        COUNT(*) AS total_observaciones,
        COUNT(*) - COUNT(VALUE) AS nulos_value,
        ROUND(100.0 * (COUNT(*) - COUNT(VALUE)) / COUNT(*), 2) AS porcentaje_nulos
    FROM observations;
""").df()

,total_observaciones,nulos_value,porcentaje_nulos
0,409803,0,0.0


In [5]:
conn.execute("""
    SELECT COUNT(*) AS encuentros_sin_paciente
    FROM encounters e
    LEFT JOIN patients p ON e.PATIENT = p.Id
    WHERE p.Id IS NULL;
""").df()

,encuentros_sin_paciente
0,10


In [6]:
conn.execute("""
SELECT 
    COUNT(*) AS total_pacientes,
    COUNT(*) FILTER (WHERE DEATHDATE IS NULL) AS pacientes_vivos,
    COUNT(DEATHDATE) AS pacientes_fallecidos
FROM patients;
""").df()

,total_pacientes,pacientes_vivos,pacientes_fallecidos
0,556,482,74


In [7]:
conn.execute("""
SELECT 
    GENDER AS sexo,
    COUNT(*) AS total_pacientes,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS porcentaje
FROM patients
GROUP BY GENDER;
""").df()

,sexo,total_pacientes,porcentaje
0,F,266,47.84
1,M,290,52.16


In [8]:
conn.execute("""
SELECT 
    DESCRIPTION AS diagnostico,
    CODE AS codigo_icd,
    COUNT(*) AS total_casos
FROM conditions
GROUP BY DESCRIPTION, CODE
ORDER BY total_casos DESC
LIMIT 10;
""").df()

,diagnostico,codigo_icd,total_casos
0,Medication review due (situation),314529007,4070
1,Stress (finding),73595000,1539
2,Gingivitis (disorder),66383009,1459
3,Full-time employment (finding),160903007,1451
4,Part-time employment (finding),160904001,922
5,Social isolation (finding),422650009,561
6,Viral sinusitis (disorder),444814009,548
7,Limited social contact (finding),423315002,518
8,Not in labor force (finding),741062008,496
9,Gingival disease (disorder),18718003,398


In [9]:
conn.execute("""
SELECT 
    Id AS patient_id,
    FIRST AS nombre,
    LAST AS apellido,
    BIRTHDATE AS fecha_nacimiento
FROM patients
WHERE BIRTHDATE < '1960-01-01'
ORDER BY BIRTHDATE ASC;
""").df()

,patient_id,nombre,apellido,fecha_nacimiento
0,b022a09d-4a99-a557-41b7-39904c569fa3,Giovanni385,Cormier289,1925-01-14
1,7edbeb52-c6ca-2738-9a05-3b6f83ab88fa,Eusebio566,Turcotte120,1925-01-14
2,179b6797-6f02-fb0c-dae8-78733b14b58d,Matt836,Anderson154,1925-01-14
3,b7f0a298-c3e9-83ab-686d-34c47e900dec,Juan88,Delvalle807,1925-07-10
4,992a40d5-dfd2-1302-90f4-4174afbce6a1,Levi940,Bartell116,1925-08-05
...,...,...,...,...
98,1f5e1f8c-268c-3063-0da4-6eb84a90a567,Fabian647,Cruickshank494,1958-11-17
99,a37ff911-f200-60d9-89e0-ad0a211127fa,Wade235,Koepp521,1958-11-21
100,0f61794e-5ec4-c7fa-d9ed-91c86b880de3,Ashley34,Pfannerstill264,1959-03-07
101,c580b7bb-62b6-a6df-cdab-0ae2b9b802a9,Floretta117,Hamill307,1959-03-26


In [10]:
conn.execute("""
SELECT 
    EXTRACT(YEAR FROM START) AS anio,
    COUNT(*) AS total_encuentros
FROM encounters
GROUP BY anio
ORDER BY anio ASC;
""").df()

,anio,total_encuentros
0,1929,2
1,1937,2
2,1940,1
3,1942,2
4,1943,7
...,...,...
83,2022,2167
84,2023,1925
85,2024,1998
86,2025,2095


#Bloque I: Preguntas básicas

1. **¿Cuántos pacientes hay? ¿Cuántos siguen vivos?** 
   La base de datos cuenta con **556 pacientes**[cite: 8]. La mayoría conforman una cohorte activa (482 pacientes vivos), representando los 74 pacientes fallecidos un ~13.3% de mortalidad histórica registrada[cite: 8].

2. **Distribución por sexo** 
   La distribución por sexo no presenta sesgos severos. Existe una ligera dominancia del sexo masculino con el **52.16%** (290 pacientes), frente al **47.84%** del femenino (266 pacientes)[cite: 8].

3. **Los 10 diagnósticos más frecuentes**
   Los datos sugieren un entorno de medicina preventiva o atención primaria[cite: 9]. El volumen abrumador del código `314529007` (*Medication review due* con 4,070 casos) indica protocolos rutinarios de seguimiento farmacéutico[cite: 9]. Las siguientes comorbilidades (*Stress* y *Gingivitis* con ~1,500 casos) apuntan a hallazgos ambulatorios de estilo de vida en lugar de urgencias agudas o especialidades complejas[cite: 9].

4. **Pacientes nacidos antes de 1960**
   Se identificaron **103 pacientes** nacidos antes del 1 de enero de 1960[cite: 10]. Algunos de estos expedientes contienen registros de fechas de nacimiento en la década de 1920 (ej. 1925)[cite: 10].

5. **Encuentros por año**
   El registro clínico abarca **88 años** de encuentros continuos[cite: 11]. Aunque existen registros desde 1929[cite: 11] se observa un salto masivo en la captura de información durante la última década, manteniendo una media de ~2,000 encuentros anuales entre 2022 y 2026[cite: 11].

In [11]:
conn.execute("""
WITH encuentros_por_paciente AS (
    SELECT 
        PATIENT,
        COUNT(*) AS total_encuentros
    FROM encounters
    GROUP BY PATIENT
)
SELECT 
    ROUND(AVG(total_encuentros), 2) AS promedio_encuentros,
    MEDIAN(total_encuentros) AS mediana_encuentros
FROM encuentros_por_paciente;
""").df()

,promedio_encuentros,mediana_encuentros
0,54.92,35.0


In [12]:
conn.execute("""
WITH primer_diagnostico AS (
    SELECT 
        c.PATIENT,
        MIN(c.START) AS fecha_primer_diag
    FROM conditions c
    WHERE LOWER(c.DESCRIPTION) LIKE '%hypertension%'
    GROUP BY c.PATIENT
)
SELECT 
    ROUND(AVG(DATEDIFF('year', p.BIRTHDATE, pd.fecha_primer_diag)), 2) AS edad_media_primer_diagnostico
FROM primer_diagnostico pd
JOIN patients p ON pd.PATIENT = p.Id;

""").df()

,edad_media_primer_diagnostico
0,43.76


In [13]:
conn.execute("""
SELECT 
    DESCRIPTION AS diagnostico,
    COUNT(DISTINCT PATIENT) AS pacientes_unicos
FROM conditions
GROUP BY DESCRIPTION
HAVING COUNT(DISTINCT PATIENT) > 50
ORDER BY pacientes_unicos DESC;
""").df()

,diagnostico,pacientes_unicos
0,Medication review due (situation),540
1,Gingivitis (disorder),431
2,Stress (finding),425
3,Full-time employment (finding),424
4,Viral sinusitis (disorder),362
5,Part-time employment (finding),346
6,Social isolation (finding),287
7,Limited social contact (finding),271
8,Received higher education (finding),258
9,Body mass index 30+ - obesity (finding),252


In [14]:
conn.execute("""
SELECT 
    DESCRIPTION AS analito,
    ROUND(AVG(TRY_CAST(VALUE AS DOUBLE)), 2) AS media,
    PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY TRY_CAST(VALUE AS DOUBLE)) AS mediana,
    PERCENTILE_CONT(0.10) WITHIN GROUP (ORDER BY TRY_CAST(VALUE AS DOUBLE)) AS p10,
    PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY TRY_CAST(VALUE AS DOUBLE)) AS p90
FROM observations
WHERE LOWER(DESCRIPTION) LIKE '%glucose%'
  AND TRY_CAST(VALUE AS DOUBLE) IS NOT NULL
GROUP BY DESCRIPTION;
""").df()

,analito,media,mediana,p10,p90
0,Glucose [Mass/volume] in Serum or Plasma,89.83,86.6,68.4,115.0
1,Glucose [Presence] in Urine by Test strip,0.32,0.0,0.0,1.0
2,Glucose [Mass/volume] in Blood,88.24,85.5,68.3,111.5
3,Glucose [Mass/volume] in Urine by Test strip,1.58,1.5,0.7,2.3


In [15]:
conn.execute("""
WITH edad_pacientes AS (
    SELECT 
        GENDER,
        DATEDIFF('year', BIRTHDATE, COALESCE(DEATHDATE, CURRENT_DATE)) AS edad
    FROM patients
)
SELECT 
    CASE 
        WHEN edad < 18 THEN '0-17 años'
        WHEN edad BETWEEN 18 AND 39 THEN '18-39 años'
        WHEN edad BETWEEN 40 AND 64 THEN '40-64 años'
        ELSE '65+ años'
    END AS grupo_etario,
    GENDER AS sexo,
    COUNT(*) AS total_pacientes
FROM edad_pacientes
GROUP BY grupo_etario, GENDER
ORDER BY grupo_etario, GENDER;
""").df()

,grupo_etario,sexo,total_pacientes
0,0-17 años,F,52
1,0-17 años,M,57
2,18-39 años,F,67
3,18-39 años,M,85
4,40-64 años,F,89
5,40-64 años,M,99
6,65+ años,F,58
7,65+ años,M,49


#Bloque II: Preguntas intermedias
**Número medio y mediano de encuentros por paciente** 
   Existe una clara asimetría en la demanda de atención. Mientras el promedio de encuentros por paciente es de **54.92**, la mediana se sitúa en **35.0**[cite: 9]. Esta diferencia (~20 encuentros) indica un sesgo positivo en la distribución: la mayoría de los pacientes asisten moderadamente, pero un subgrupo de hiperfrecuentadores con enfermedades crónicas o seguimientos largos tracciona el promedio hacia arriba[cite: 9].

2. **Edad media al primer diagnóstico de una condición que elijas (hipertensión)** 
   La edad media al primer diagnóstico de hipertensión esencial en esta cohorte es de **43.76 años**[cite: 10]. Esto subraya la importancia del tamizaje cardiovascular a partir de la cuarta década de la vida, alineándose con las guías de medicina preventiva[cite: 10].

3. **Diagnósticos con más de 50 pacientes distintos** 
   Al evaluar las condiciones presentes en más de 50 pacientes, encontramos 53 categorías distintas[cite: 11]. Se confirma el perfil de atención primaria, dominado por revisiones farmacológicas (540 pacientes), gingivitis (431 pacientes) y estrés (425 pacientes)[cite: 11].

4. **Distribución de un analito (media, mediana, p10, p90) (glucosa)** 
   El análisis poblacional de la glucosa en suero/plasma muestra un excelente control glucémico general. La mediana es de **86.6 mg/dL** y la media de **89.83 mg/dL**[cite: 12]. Incluso en el decil superior (Percentil 90), el valor es de **115.0 mg/dL**, indicando que el 90% de las tomas se mantienen por debajo del umbral diagnóstico de diabetes y apenas entran en rango de prediabetes[cite: 12].

5. **Pacientes por grupo etario y sexo** 
   La mayor carga de pacientes se concentra en el bloque de adultos de **40-64 años** (188 pacientes en total), con una ligera predominancia masculina (99 vs 89)[cite: 13]. Sin embargo, en el bloque geriátrico (65+ años), la tendencia se invierte y las mujeres (58) superan a los hombres (49), un reflejo demográfico clásico de la mayor esperanza de vida femenina[cite: 13].

In [16]:
conn.execute("""
SELECT 
    p.Id AS patient_id,
    p.FIRST AS nombre,
    p.LAST AS apellido
FROM patients p
JOIN conditions c1 ON p.Id = c1.PATIENT AND LOWER(c1.DESCRIPTION) LIKE '%diabetes%'
JOIN conditions c2 ON p.Id = c2.PATIENT AND LOWER(c2.DESCRIPTION) LIKE '%hypertension%'
GROUP BY p.Id, p.FIRST, p.LAST;
""").df()

,patient_id,nombre,apellido
0,703030ae-3aea-06f6-a392-e95a3312cf99,Dianna917,Grady603
1,07c8d48c-adae-5755-6f73-5768743fcde4,Fran41,Ledner144
2,06d34e6d-e4ea-34de-82a9-15d395916f8a,Kristal814,Simonis280
3,de276e36-b2b5-e215-fbb4-e6355242d4ce,Deandrea896,Osinski784
4,17051f39-218c-e6b4-f097-0b71f166136b,Jeanie708,Howell947
...,...,...,...
87,2951cb9a-a0cc-ba5d-50e5-0f7ded4fa37b,Neta452,Spinka232
88,6138c0fb-a664-2786-c443-ce3dedfd8d06,Amalia471,Uribe503
89,c9f67b8f-f768-17d8-1a42-3e1fdd3df850,Francina926,Beer512
90,2f97b589-4702-52d6-a569-f4641a7f6453,Blair400,Zemlak964


In [17]:
conn.execute("""
SELECT DISTINCT
    p.Id AS patient_id,
    p.FIRST AS nombre,
    p.LAST AS apellido
FROM patients p
JOIN conditions c ON p.Id = c.PATIENT
WHERE LOWER(c.DESCRIPTION) LIKE '%diabetes%'
  AND NOT EXISTS (
      SELECT 1 
      FROM observations o 
      WHERE o.PATIENT = p.Id 
        AND (LOWER(o.DESCRIPTION) LIKE '%hba1c%' OR LOWER(o.DESCRIPTION) LIKE '%hemoglobin a1c%')
  );
""").df()

,patient_id,nombre,apellido
0,07c8d48c-adae-5755-6f73-5768743fcde4,Fran41,Ledner144
1,3cb59357-5d6e-bb70-e7d2-2b16ee0357d6,Roberto515,Santiago500
2,cbe9f5e1-165a-9507-d904-f0fa486ea1c5,Rex53,Murphy561
3,5bb9328e-56f5-a14c-7fdf-3911be4be7bc,Benton624,Harris789
4,4358663f-ee77-411b-cf3f-f2f5ee07c18d,Max124,Ziemann98
5,aa08743c-9520-8b20-5072-e6b40de9e5aa,Leo278,Walter473
6,5b6a19a6-e5fb-44d4-4646-5f9daf911e17,Raymundo71,Tromp100
7,4b2cc620-5e18-f943-cf43-212e55047402,Zachary28,Rolfson709
8,3584c5f8-85da-a91a-6e7e-5df8753189e0,Jerry654,Orn563
9,b9c35397-c3a9-bc5b-5dc0-818f5d5f0944,Cherise743,Rodriguez71


In [18]:
conn.execute("""
SELECT 
    PATIENT AS patient_id,
    MIN(START) AS primer_encuentro,
    MAX(START) AS ultimo_encuentro,
    DATEDIFF('day', MIN(START), MAX(START)) AS dias_entre_primer_y_ultimo
FROM encounters
GROUP BY PATIENT;
""").df()

,patient_id,primer_encuentro,ultimo_encuentro,dias_entre_primer_y_ultimo
0,7e4ca2a6-ac03-266d-996e-2d0d59ed55c6,2018-12-11 06:15:05,2025-12-09 06:15:05,2555
1,32d77305-8ed6-dc9f-9d9b-fdcef02809dc,2009-01-08 19:30:20,2026-09-17 19:30:20,6461
2,971c6cb5-4c81-f0e8-39f8-c52cb46a3828,1995-08-05 13:57:38,2025-07-19 13:57:38,10941
3,3acf1284-0887-d559-1b15-698380374161,1975-12-21 13:38:54,2026-03-30 21:38:54,18362
4,fe2a0ff9-09e4-011b-ecdf-b4bbe1686076,2014-09-10 16:26:44,2026-06-06 08:26:44,4287
...,...,...,...,...
542,27a9cdbc-6dd2-8417-2a47-3682de85c386,2001-09-16 13:32:36,2025-08-03 13:32:36,8722
543,b8eb8e01-8ebd-5cfd-e515-7ddfb755a09c,1989-01-05 15:43:41,2025-10-10 17:43:41,13427
544,2abee97c-9072-3a9c-d34f-d2cc33d9b686,2011-04-06 21:39:58,2026-05-02 18:47:53,5505
545,4b002960-b79d-08d6-f48e-09d2e0fe6b91,1984-09-01 16:08:49,2026-07-02 09:09:36,15279


In [19]:
conn.execute("""
SELECT 
    m.DESCRIPTION AS medicamento,
    COUNT(DISTINCT m.PATIENT) AS pacientes_prescritos,
    COUNT(*) AS total_prescripciones
FROM medications m
JOIN conditions c ON m.PATIENT = c.PATIENT
WHERE LOWER(c.DESCRIPTION) LIKE '%diabetes%'
GROUP BY m.DESCRIPTION
ORDER BY total_prescripciones DESC
LIMIT 1;
""").df()

,medicamento,pacientes_prescritos,total_prescripciones
0,insulin isophane human 70 UNT/ML / insulin r...,43,13156


In [20]:
conn.execute("""
SELECT DISTINCT
    e.PATIENT AS patient_id,
    e.Id AS encounter_id,
    e.START AS fecha_urgencia,
    e.ENCOUNTERCLASS AS clase_encuentro
FROM encounters e
LEFT JOIN conditions c ON e.Id = c.ENCOUNTER
WHERE LOWER(e.ENCOUNTERCLASS) IN ('emergency', 'urgentcare')
  AND c.CODE IS NULL;
""").df()

,patient_id,encounter_id,fecha_urgencia,clase_encuentro
0,8999ee3b-539c-9edd-8ed9-172187f4e4f8,8999ee3b-539c-9edd-14a4-3a8829a13fb6,2006-03-31 06:44:45,emergency
1,fb0f4069-2f65-7830-1ec9-f5f73454923d,fb0f4069-2f65-7830-074e-8b28b8adf576,2019-04-29 23:14:36,urgentcare
2,acb826d5-6877-d18a-cddf-e2c05d135d8c,acb826d5-6877-d18a-c0dc-aca4e4fbc590,2017-08-04 20:14:43,emergency
3,6cee92c0-efd9-e141-3b1f-cab92c843c7c,6cee92c0-efd9-e141-bef8-2746e760a0db,2025-03-21 18:44:51,emergency
4,4d409bf8-aed5-7c15-078d-b3ca9b04dcaa,4d409bf8-aed5-7c15-764c-aaa9cedbe9a5,2020-08-13 03:49:44,emergency
...,...,...,...,...
825,f3f263c3-35be-8bcf-c3e3-728aafd0f92d,f3f263c3-35be-8bcf-3345-bba6296a7098,2008-12-22 13:01:04,emergency
826,1f5e1f8c-268c-3063-0da4-6eb84a90a567,1f5e1f8c-268c-3063-6090-830bf3cb1bf3,2024-06-03 08:37:45,urgentcare
827,17f4f654-c228-6171-109b-82d23b516767,17f4f654-c228-6171-563a-12bc47bbd6cd,2022-02-13 07:03:39,emergency
828,dbaa9945-59a6-ecec-9d2f-b85f62301bcf,dbaa9945-59a6-ecec-1c94-606444331944,2019-07-13 20:30:08,urgentcare


#Bloque III: Preguntas avanzadas
1. **Pacientes con diabetes y hipertensión** 
   El análisis revela que **92 pacientes** padecen simultáneamente diabetes e hipertensión[cite: 14]. Esta cohorte representa un grupo de alto riesgo cardiovascular que requiere monitoreo clínico y farmacológico estrecho.

2. **Pacientes con diabetes sin ninguna medición de HbA1c** 
   Se identificó un grupo de pacientes con diagnóstico de diabetes que **no cuentan con ninguna medición registrada de Hemoglobina Glucosilada (HbA1c)**[cite: 14]. Clínicamente, esto representa una falla grave en el protocolo de seguimiento estándar, o bien, una brecha en la captura de resultados de laboratorio dentro del sistema.

3. **Para cada paciente: su primer y su último encuentro** 
   El cálculo de días entre el primer y último encuentro muestra expedientes con un seguimiento de vida completo. El caso más extremo documenta **35,188 días** de diferencia (más de 96 años de historia clínica, con encuentros desde 1929 hasta 2026)[cite: 15]. Esto valida a los 547 pacientes que tienen registros de actividad en la tabla de encuentros[cite: 15].

4. **Medicamento más frecuente entre los pacientes de una condición** 
   El medicamento de mayor volumen para diabéticos es la **insulina isófana humana (70/30)**[cite: 15]. Lo más destacable es la densidad de resurtimiento: se generaron **13,156 prescripciones** de este fármaco, pero concentradas en tan solo **43 pacientes únicos**[cite: 15]. Esto demuestra el intenso requerimiento de dispensación periódica que demanda el tratamiento insulínico.

5. **Pacientes que tuvieron un encuentro de urgencias sin diagnóstico registrado** 
   Se detectaron **830 encuentros** clasificados como urgencias (`emergency` o `urgentcare`) que finalizaron **sin un diagnóstico asociado** en la tabla de condiciones (`c.CODE IS NULL`)[cite: 15]. Desde la perspectiva de ingeniería y calidad de datos, esto es una alerta, ya que todo egreso de un servicio de urgencias debería justificar clínicamente su motivo.

In [21]:
#Actividad 7: El join que multiplica
#1. Número base (correcto)
print("1. Todos los encuentros:")
display(conn.execute("""
    SELECT 
        COUNT(*) AS total_encuentros,
        ROUND(AVG(DATEDIFF('minute', START, STOP)), 2) AS duracion_promedio_minutos
    FROM encounters;
""").df())

#2. El error (join)
print("\n2. Encuentros duplucados (error):")
display(conn.execute("""
    SELECT 
        COUNT(*) AS total_encuentros_duplicados,
        ROUND(AVG(DATEDIFF('minute', e.START, e.STOP)), 2) AS promedio_falso_minutos
    FROM encounters e
    JOIN conditions c ON e.Id = c.ENCOUNTER;
""").df())

#3. Correción
print("\n3. Encuentros corregidos:")
display(conn.execute("""
    SELECT 
        COUNT(*) AS total_encuentros_corregidos,
        ROUND(AVG(DATEDIFF('minute', e.START, e.STOP)), 2) AS promedio_verdadero_minutos
    FROM encounters e
    WHERE EXISTS (
        SELECT 1 
        FROM conditions c 
        WHERE c.ENCOUNTER = e.Id
    );
""").df())

1. Todos los encuentros:


,total_encuentros,duracion_promedio_minutos
0,30042,374.24



2. Encuentros duplucados (error):


,total_encuentros_duplicados,promedio_falso_minutos
0,19907,148.41



3. Encuentros corregidos:


,total_encuentros_corregidos,promedio_verdadero_minutos
0,12806,164.96


La relación entre encounters y conditions es de 1 a muchos (1:N). Un paciente en una consulta de rutina puede recibir un solo diagnóstico, pero un paciente ingresado en urgencias puede salir con uno o más. Al usar un join directo, el motor de SQL no agrupa; multiplica. El encuentro de urgencias de esa persona se clona 5 veces en la tabla resultante. El count(*) inflado ya no cuenta "encuentros", cuenta "diagnósticos asociados a encuentros".
Al clonarse las filas de los pacientes con múltiples diagnósticos, el promedio pondera de forma asimétrica los encuentros complejos.
Si un encuentro de urgencias con 5 diagnósticos duró 800 minutos, esos 800 minutos se suman 5 veces al cálculo del promedio. Las consultas sencillas de 15 minutos (1 diagnóstico) se suman una sola vez.
WHERE EXISTS actúa como un filtro binario. Con esto lo que hacemos es que revisamos si el enceuntro tiene al menos un registro en la tabla de conditions, si es cierto devuelve el encuentro una sola vez y ya no continua. Con esto logramos mantener la integridad de la fila y calculamos la duración verdadera.

En el resultado podemos ver lo siguiente:
Todos los encuentros (30,042): Incluye absolutamente todo, desde urgencias de larga duración hasta alguien que solo pasó a recoger una receta o ponerse una vacuna y no se le diagnosticó ninguna enfermedad nueva.
Encuentros duplicados (19,907): El join hizo dos cosas; eliminó a los pacientes sin diagnóstico y clonó los encuentros. En esos 12,806 encuentros reales, se emitieron 19,907 diagnósticos en total. Al clonar las filas para que cada diagnóstico tuviera su propio encuentro, el promedio se alteró bajando a 148.41 minutos.   
Encuentros corregidos (12,806): Este es tu subgrupo real. De los 30,042 encuentros totales, solamente en 12,806 el paciente recibió un diagnóstico. El promedio real de una cita donde sí te diagnostican algo es de 164.96 minutos.   2

In [22]:
#Actividad 8: La trampa del LEFT JOIN
#1. Filtro en el WHERE
print("1. Filtro de la tabla derecha en el WHERE:")
display(conn.execute("""
    SELECT COUNT(*) AS filas_totales
    FROM patients p
    LEFT JOIN encounters e ON p.Id = e.PATIENT
    WHERE e.ENCOUNTERCLASS = 'emergency';
""").df())

#2. Filtro en ON (correcto)
print("\n2. Filtro de la tabla derecha en el ON:")
display(conn.execute("""
    SELECT COUNT(*) AS filas_totales
    FROM patients p
    LEFT JOIN encounters e ON p.Id = e.PATIENT AND e.ENCOUNTERCLASS = 'emergency';
""").df())

1. Filtro de la tabla derecha en el WHERE:


,filas_totales
0,1182



2. Filtro de la tabla derecha en el ON:


,filas_totales
0,1350


Al poner el filtro en el WHERE el motor de SQL primero ejecuta el LEFT JOIN, uniendo a los pacientes con sus encuentros y rellenando con NULL los datos de los pacientes que no tienen encuentros. Despues, evalua WHERE e.ENCOUNTERCLASS = 'emergency', y cualquier paceinte que tenga NULL en esa columna es descartado, lo que provoca que el LEFT JOIN se comporte como un INNER JOIN, sobreviviendo solo los pacientes que fueron a urgencias. AHora cuando se coloca el JOIN en ON le indicamos a DuckDB que traiga todos los registros de urgencias para cada paciente, y si no tiene urgencias aun así lo cruza con NULL porque aplica el LEFT JOIN, manteniendo así todos los pacientes de la base clínica, tanto los que recibieron urgencias como los que no (con su NULL)

In [23]:
# Actividad 9: Medir el rendimiento
#1. Consulta sub-óptima
print("Primer plan")
plan_lento = conn.execute("""
    EXPLAIN ANALYZE 
    WITH todo_mezclado AS (
        SELECT p.*, c1.DESCRIPTION AS diag1, c2.DESCRIPTION AS diag2
        FROM patients p
        JOIN conditions c1 ON p.Id = c1.PATIENT
        JOIN conditions c2 ON p.Id = c2.PATIENT
    )
    SELECT Id, FIRST, LAST
    FROM todo_mezclado
    WHERE LOWER(diag1) LIKE '%diabetes%' 
      AND LOWER(diag2) LIKE '%hypertension%';
""").fetchall()

print(plan_lento[0][1])


#2. Consulta optimizada
print("\n2. Segundo plan")
plan_rapido = conn.execute("""
    EXPLAIN ANALYZE 
    SELECT p.Id, p.FIRST, p.LAST
    FROM patients p
    JOIN conditions c1 ON p.Id = c1.PATIENT AND LOWER(c1.DESCRIPTION) LIKE '%diabetes%'
    JOIN conditions c2 ON p.Id = c2.PATIENT AND LOWER(c2.DESCRIPTION) LIKE '%hypertension%'
    GROUP BY p.Id, p.FIRST, p.LAST;
""").fetchall()

print(plan_rapido[0][1])

Primer plan
╭─ Summary ───────────╮
│ Total Time: 0.0020s │
╰─────────────────────╯
╭─ Projection ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Projections: Id, FIRST, LAST                                                                                                                                      │
│ 221 rows                                                                                                                                                      0µs │
╰─────────────────────────────────────────────────────────────────────────────────┰─────────────────────────────────────────────────────────────────────────────────╯
╭─ Hash Join ─────────────────────────────────────────────────────────────────────┸─────────────────────────────────────────────────────────────────────────────────╮
│ Join Type: INNER                                                                    

##Actividad 9

**Tiempos de ejecución:**
* **Primer plan:** 0.0036 segundos (3.6 ms).
* **Segundo plan:** 0.0030 segundos (3.0 ms).

**¿Qué cambió?:**
* **La corrección de la cardinalidad:** El primer plan procesa y arroja 221 filas en su proyección final debido al efecto multiplicador (*fan-out*) de cruzar dos veces la tabla de condiciones. El segundo plan introduce un paso de `Hash Group By` que agrupa correctamente los datos y devuelve a los 92 pacientes únicos, reduciendo la carga final de memoria.
* **La inteligencia del optimizador columnar:** Aunque escribimos un `SELECT *` y un filtro externo en la primera consulta, el árbol demuestra que DuckDB reescribió la consulta por debajo. En ambos planes vemos `Projections: Id, FIRST, LAST` y filtros dinámicos empujados directamente al `Table Scan`. El motor analítico corrigió nuestra mala sintaxis automáticamente (*Projection Pushdown* y *Predicate Pushdown*).

**¿Por qué un índice (B-Tree) no ayuda aquí?**
DuckDB utiliza `Sequential Scan` en todas las tablas. En motores OLTP como PostgreSQL, un escaneo secuencial es lento y requiere la creación de índices B-Tree para saltar directo al dato. En DuckDB (OLAP), los índices clásicos casi no se utilizan por el diseño columnar:
1. **Pruning de columnas (I/O mínimo):** Al guardar los datos en disco separados por columnas (y no en bloques de filas completas), el motor ignora físicamente todas las columnas que no se pidieron. Un índice no aporta mucho porque el disco ya lee lo mínimo necesario.
2. **Data skipping (Zone Maps):** En lugar de índices, DuckDB mantiene estadísticas ligeras (min/max) de cada bloque columnar. Cuando aplicamos el filtro `LOWER(DESCRIPTION) LIKE '%diabetes%'`, el motor lee las estadísticas y se salta bloques enteros en microsegundos.
3. **Ejecución vectorizada:** El escaneo secuencial en DuckDB no lee fila por fila; lee vectores de miles de valores al mismo tiempo y los procesa directamente en la caché del CPU, superando la velocidad de lectura de un índice tradicional.

In [24]:
#Actividad 10: SQL contra pandas
import pandas as pd
import time

df_patients = conn.execute("SELECT * FROM patients").df()
df_conditions = conn.execute("SELECT * FROM conditions").df()

print("Comparativa: SQL vs Pandas")

#Consulta 1 (básica): distribución por sexo
start_time = time.time()
q1 = df_patients.groupby('GENDER').size().reset_index(name='total_pacientes')
q1['porcentaje'] = round((q1['total_pacientes'] / len(df_patients)) * 100, 2)
tiempo_q1 = time.time() - start_time
print(f"1. Pandas (distribución aexo): {tiempo_q1:.5f} segundos")

#Consulta 2 (intermedia): diagnósticos > 50 pacientes
start_time = time.time()
q2 = df_conditions.groupby('DESCRIPTION')['PATIENT'].nunique().reset_index(name='pacientes_unicos')
q2 = q2[q2['pacientes_unicos'] > 50].sort_values('pacientes_unicos', ascending=False)
tiempo_q2 = time.time() - start_time
print(f"2. Pandas (filtro en agrupación): {tiempo_q2:.5f} segundos")

#Consulta 3 (avanzada): comorbilidad diabetes + hipertensión
start_time = time.time()
diab = df_conditions[df_conditions['DESCRIPTION'].str.lower().str.contains('diabetes', na=False)]
hyp = df_conditions[df_conditions['DESCRIPTION'].str.lower().str.contains('hypertension', na=False)]
comorb = pd.merge(diab, hyp, on='PATIENT', how='inner')
q3 = pd.merge(df_patients, comorb, left_on='Id', right_on='PATIENT', how='inner')[['Id', 'FIRST', 'LAST']].drop_duplicates()
tiempo_q3 = time.time() - start_time
print(f"3. Pandas (cruces complejos): {tiempo_q3:.5f} segundos")

Comparativa: SQL vs Pandas
1. Pandas (distribución aexo): 0.00369 segundos
2. Pandas (filtro en agrupación): 0.00606 segundos
3. Pandas (cruces complejos): 0.01262 segundos


Para el volumen de 556 pacientes, aunque pandas resuelve en 0.01331 segundos (13.3 ms) el cruce complejo, SQL lo resolvió el mismo cruce ~4 veces más rapido (3 ms).
Utilizaría Pandas al salir de la fase de extracción y filtrado, cuando el análisis requiera una imputación compleja, modelado estadistico o preparación de tensores para ML.